# Amazon Sales Data Analysis — Machine Learning

Builds and evaluates classification models for predicting high-sales products using features from the Amazon products dataset.


In [45]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils import resample

In [46]:
df = pd.read_csv("amazon_products.csv")
print(df.head())
print(df.columns.tolist())
print(df.shape)

                                       product_title  product_rating  \
0  BOYA BOYALINK 2 Wireless Lavalier Microphone f...             4.6   
1  LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...             4.3   
2  DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...             4.6   
3  Apple AirPods Pro 2 Wireless Earbuds, Active N...             4.6   
4  Apple AirTag 4 Pack. Keep Track of and find Yo...             4.8   

   total_reviews  purchased_last_month  discounted_price  original_price  \
0          375.0                 300.0             89.68          159.00   
1         2457.0                6000.0              9.99           15.99   
2         3044.0                2000.0            314.00          349.00   
3        35882.0               10000.0            162.24          162.24   
4        28988.0               10000.0             72.74           72.74   

  is_best_seller is_sponsored             has_coupon buy_box_availability  \
0       No Badge    Sponsored  Sa

In [47]:
def find_column(columns, options):
    lower_map = {col.lower().strip(): col for col in columns}
    for option in options:
        if option.lower().strip() in lower_map:
            return lower_map[option.lower().strip()]
    for col in columns:
        col_clean = col.lower().strip()
        for option in options:
            if option.lower().strip() in col_clean:
                return col
    return None

sales_col = find_column(df.columns, [
    "purchased_last_month",
    "units_purchased_last_month",
    "bought in past month",
    "bought in the last month",
    "purchases_last_month"
])

rating_col = find_column(df.columns, [
    "rating",
    "ratings"
])

review_col = find_column(df.columns, [
    "rating_count",
    "review_count",
    "reviews",
    "no_of_ratings",
    "rating count",
    "number_of_reviews"
])

discount_col = find_column(df.columns, [
    "discount_percentage",
    "discount percent",
    "discount"
])

price_col = find_column(df.columns, [
    "discounted_price",
    "price",
    "selling_price"
])

original_price_col = find_column(df.columns, [
    "actual_price",
    "original_price",
    "mrp"
])

category_col = find_column(df.columns, [
    "sub_category",
    "category",
    "main_category"
])

sponsored_col = find_column(df.columns, [
    "is_sponsored",
    "sponsored",
    "sponsored_label"
])

print("sales_col:", sales_col)
print("rating_col:", rating_col)
print("review_col:", review_col)
print("discount_col:", discount_col)
print("price_col:", price_col)
print("original_price_col:", original_price_col)
print("category_col:", category_col)
print("sponsored_col:", sponsored_col)

sales_col: purchased_last_month
rating_col: product_rating
review_col: total_reviews
discount_col: discount_percentage
price_col: discounted_price
original_price_col: original_price
category_col: product_category
sponsored_col: is_sponsored


In [48]:
def clean_numeric(series):
    return pd.to_numeric(
        series.astype(str).str.replace(r"[^0-9.]", "", regex=True),
        errors="coerce"
    )

if sales_col:
    df[sales_col] = clean_numeric(df[sales_col])

if rating_col:
    df[rating_col] = clean_numeric(df[rating_col])

if review_col:
    df[review_col] = clean_numeric(df[review_col])

if discount_col:
    df[discount_col] = clean_numeric(df[discount_col])

if price_col:
    df[price_col] = clean_numeric(df[price_col])

if original_price_col:
    df[original_price_col] = clean_numeric(df[original_price_col])

print(df[[col for col in [sales_col, rating_col, review_col, discount_col, price_col, original_price_col] if col is not None]].head())

   purchased_last_month  product_rating  total_reviews  discount_percentage  \
0                 300.0             4.6          375.0                43.60   
1                6000.0             4.3         2457.0                37.52   
2                2000.0             4.6         3044.0                10.03   
3               10000.0             4.6        35882.0                 0.00   
4               10000.0             4.8        28988.0                 0.00   

   discounted_price  original_price  
0             89.68          159.00  
1              9.99           15.99  
2            314.00          349.00  
3            162.24          162.24  
4             72.74           72.74  


In [49]:
median_sales = df[sales_col].median()
df["high_sales"] = (df[sales_col] > median_sales).astype(int)

print("Median sales:", median_sales)
print(df["high_sales"].value_counts())
print(df["high_sales"].value_counts(normalize=True))

Median sales: 200.0
high_sales
0    30924
1    11751
Name: count, dtype: int64
high_sales
0    0.72464
1    0.27536
Name: proportion, dtype: float64


In [50]:
#Prediction Task

print("The outcome variable for this project is high_sales.")

print("1 means the product had high sales, meaning its number of units purchased in the last month was above the median value in the dataset.")
print("0 means the product had low sales, meaning its number of units purchased in the last month was at or below the median value in the dataset.")

print("I chose this outcome because the main goal of the project is to understand what product characteristics are connected to stronger sales performance.")

The outcome variable for this project is high_sales.
1 means the product had high sales, meaning its number of units purchased in the last month was above the median value in the dataset.
0 means the product had low sales, meaning its number of units purchased in the last month was at or below the median value in the dataset.
I chose this outcome because the main goal of the project is to understand what product characteristics are connected to stronger sales performance.


In [51]:
# Predictor Variables

print("The predictor variables used in this model are:")

print("discount percentage: products with bigger discounts may attract more buyers.")
print("rating: products with higher ratings may be seen as better quality.")
print("review count: products with more reviews may look more trustworthy and popular.")
print("discounted price: lower selling prices may make products easier to buy.")
print("original price: this helps show how much the item originally cost before the discount.")
print("category: some categories may naturally sell more than others.")
print("sponsored status: sponsored products may get more visibility.")
print("price gap: the difference between original price and discounted price may show how strong the deal looks to customers.")

The predictor variables used in this model are:
discount percentage: products with bigger discounts may attract more buyers.
rating: products with higher ratings may be seen as better quality.
review count: products with more reviews may look more trustworthy and popular.
discounted price: lower selling prices may make products easier to buy.
original price: this helps show how much the item originally cost before the discount.
category: some categories may naturally sell more than others.
sponsored status: sponsored products may get more visibility.
price gap: the difference between original price and discounted price may show how strong the deal looks to customers.


In [52]:
df_model = df.copy()

if original_price_col is not None and price_col is not None:
    df_model["price_gap"] = df_model[original_price_col] - df_model[price_col]
else:
    df_model["price_gap"] = np.nan

selected_cols = ["high_sales"]

for col in [discount_col, rating_col, review_col, price_col, original_price_col, "price_gap", category_col, sponsored_col]:
    if col is not None:
        selected_cols.append(col)

df_model = df_model[selected_cols].copy()

for col in df_model.columns:
    if df_model[col].dtype == "object":
        df_model[col] = df_model[col].fillna("Missing")
    else:
        df_model[col] = df_model[col].fillna(df_model[col].median())

categorical_cols = df_model.select_dtypes(include=["object"]).columns.tolist()
categorical_cols = [col for col in categorical_cols if col != "high_sales"]

df_model = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

print(df_model.head())
print(df_model.shape)

   high_sales  discount_percentage  product_rating  total_reviews  \
0           1                43.60             4.6          375.0   
1           1                37.52             4.3         2457.0   
2           1                10.03             4.6         3044.0   
3           1                 0.00             4.6        35882.0   
4           1                 0.00             4.8        28988.0   

   discounted_price  original_price  price_gap  \
0             89.68          159.00      69.32   
1              9.99           15.99       6.00   
2            314.00          349.00      35.00   
3            162.24          162.24       0.00   
4             72.74           72.74       0.00   

   product_category_Chargers & Cables  product_category_Gaming  \
0                               False                    False   
1                               False                    False   
2                               False                    False   
3                   

In [53]:
X = df_model.drop("high_sales", axis=1)
y = df_model["high_sales"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (42675, 21)
y shape: (42675,)


In [54]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("Training outcome counts:")
print(y_train.value_counts())
print("Test outcome counts:")
print(y_test.value_counts())

Training set shape: (34140, 21)
Test set shape: (8535, 21)
Training outcome counts:
high_sales
0    24739
1     9401
Name: count, dtype: int64
Test outcome counts:
high_sales
0    6185
1    2350
Name: count, dtype: int64


In [55]:
train_data = pd.concat([X_train, y_train], axis=1)

majority_class = train_data[train_data["high_sales"] == train_data["high_sales"].mode()[0]]
minority_class = train_data[train_data["high_sales"] != train_data["high_sales"].mode()[0]]

minority_upsampled = resample(
    minority_class,
    replace=True,
    n_samples=len(majority_class),
    random_state=42
)

train_balanced = pd.concat([majority_class, minority_upsampled])

X_train_balanced = train_balanced.drop("high_sales", axis=1)
y_train_balanced = train_balanced["high_sales"]

print("Balanced training outcome counts:")
print(y_train_balanced.value_counts())

Balanced training outcome counts:
high_sales
0    24739
1    24739
Name: count, dtype: int64


In [56]:
scaler = StandardScaler()

X_train_balanced_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)

In [57]:
baseline_model = LogisticRegression(max_iter=5000)

cv_results = cross_validate(
    baseline_model,
    X_train_balanced_scaled,
    y_train_balanced,
    cv=5,
    scoring=["accuracy", "precision", "recall", "f1"]
)

print("Baseline Logistic Regression 5-Fold CV Results")
print("Average Accuracy:", cv_results["test_accuracy"].mean())
print("Average Precision:", cv_results["test_precision"].mean())
print("Average Recall:", cv_results["test_recall"].mean())
print("Average F1:", cv_results["test_f1"].mean())

Baseline Logistic Regression 5-Fold CV Results
Average Accuracy: 0.7735761898478895
Average Precision: 0.7610591325654092
Average Recall: 0.7976067577436428
Average F1: 0.7788807021492705


In [58]:
baseline_model.fit(X_train_balanced_scaled, y_train_balanced)
y_pred_baseline = baseline_model.predict(X_test_scaled)

baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
baseline_precision = precision_score(y_test, y_pred_baseline)
baseline_recall = recall_score(y_test, y_pred_baseline)
baseline_f1 = f1_score(y_test, y_pred_baseline)

print("Baseline Test Accuracy:", baseline_accuracy)
print("Baseline Test Precision:", baseline_precision)
print("Baseline Test Recall:", baseline_recall)
print("Baseline Test F1:", baseline_f1)

Baseline Test Accuracy: 0.7560632688927944
Baseline Test Precision: 0.5389308541545613
Baseline Test Recall: 0.7893617021276595
Baseline Test F1: 0.6405386740331491


In [59]:
baseline_coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": baseline_model.coef_[0]
}).sort_values(by="Coefficient", ascending=False)

print(baseline_coef_df)

                                 Feature  Coefficient
2                          total_reviews     1.231713
1                         product_rating     0.514371
12               product_category_Phones     0.458355
13    product_category_Power & Batteries     0.382642
19            product_category_Wearables     0.288427
0                    discount_percentage     0.209498
6     product_category_Chargers & Cables     0.168371
5                              price_gap     0.143764
9               product_category_Laptops     0.117470
20                is_sponsored_Sponsored     0.063374
8            product_category_Headphones     0.019167
14  product_category_Printers & Scanners     0.015997
15           product_category_Smart Home    -0.002593
16             product_category_Speakers    -0.003040
11    product_category_Other Electronics    -0.026279
18         product_category_TV & Display    -0.052561
17              product_category_Storage    -0.060068
10           product_categor

In [60]:
# Baseline Model Interpretation

print("Looking at the baseline logistic regression model, I noticed that total_reviews had the largest positive coefficient. This means that products with more reviews are more likely to be classified as high sales products. In the context of this dataset, that makes sense because products with many reviews usually have more customer attention and trust, which can lead to more purchases.")
print("I also noticed that product_rating had a positive coefficient. This suggests that higher rated products are more likely to have high sales. A possible reason is that customers may be more willing to buy products that already have strong ratings from other buyers.")
print("On the other hand, discounted_price had a strong negative coefficient. This means that as the discounted price increases, the product becomes less likely to be in the high sales group. In this dataset, that suggests lower priced items tend to sell better than more expensive ones.")
print("Similarly, original_price also had a negative coefficient. This shows that products with higher original prices were also less likely to have high sales. I think this is reasonable because more expensive items are usually bought less often than cheaper ones.")
print("Overall, the model seems to capture some important patterns in the dataset. Features related to customer feedback, like total reviews and rating, appear to increase the chance of high sales, while higher prices lower that chance. This suggests that both product reputation and affordability are important when predicting whether an item will have high sales.")

Looking at the baseline logistic regression model, I noticed that total_reviews had the largest positive coefficient. This means that products with more reviews are more likely to be classified as high sales products. In the context of this dataset, that makes sense because products with many reviews usually have more customer attention and trust, which can lead to more purchases.
I also noticed that product_rating had a positive coefficient. This suggests that higher rated products are more likely to have high sales. A possible reason is that customers may be more willing to buy products that already have strong ratings from other buyers.
On the other hand, discounted_price had a strong negative coefficient. This means that as the discounted price increases, the product becomes less likely to be in the high sales group. In this dataset, that suggests lower priced items tend to sell better than more expensive ones.
Similarly, original_price also had a negative coefficient. This shows t

In [61]:
l1_model = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    max_iter=5000,
    C=0.5
)

l1_model.fit(X_train_balanced_scaled, y_train_balanced)
y_pred_l1 = l1_model.predict(X_test_scaled)

l1_accuracy = accuracy_score(y_test, y_pred_l1)
l1_precision = precision_score(y_test, y_pred_l1)
l1_recall = recall_score(y_test, y_pred_l1)
l1_f1 = f1_score(y_test, y_pred_l1)

print("L1 Test Accuracy:", l1_accuracy)
print("L1 Test Precision:", l1_precision)
print("L1 Test Recall:", l1_recall)
print("L1 Test F1:", l1_f1)

L1 Test Accuracy: 0.7528998242530756
L1 Test Precision: 0.5347162201094785
L1 Test Recall: 0.7897872340425532
L1 Test F1: 0.6376911183645422


In [62]:
l1_coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": l1_model.coef_[0]
}).sort_values(by="Coefficient", ascending=False)

print(l1_coef_df)

                                 Feature  Coefficient
2                          total_reviews     1.224375
1                         product_rating     0.514518
12               product_category_Phones     0.461477
5                              price_gap     0.433992
13    product_category_Power & Batteries     0.384916
19            product_category_Wearables     0.289038
0                    discount_percentage     0.209157
6     product_category_Chargers & Cables     0.169680
9               product_category_Laptops     0.120399
20                is_sponsored_Sponsored     0.062876
8            product_category_Headphones     0.020257
14  product_category_Printers & Scanners     0.016686
3                       discounted_price     0.000000
15           product_category_Smart Home    -0.001506
16             product_category_Speakers    -0.001904
11    product_category_Other Electronics    -0.023373
18         product_category_TV & Display    -0.050540
17              product_cate

In [63]:
removed_features = l1_coef_df[np.isclose(l1_coef_df["Coefficient"], 0)]["Feature"].tolist()
print(removed_features)

['discounted_price']


In [64]:
# L1 Regularization Interpretation

print("Looking at the L1 model results, the model achieved an accuracy of about 0.75, which is very similar to the baseline model. The recall is relatively high at around 0.79, meaning the model does a good job identifying products that actually have high sales. However, the precision is lower at about 0.53, which suggests that when the model predicts high sales, it is not always correct.")
print("One important thing about L1 regularization is that it removes less useful features by shrinking some coefficients to 0. In my model, the feature that was removed was discounted_price.")
print("This suggests that discounted_price was not adding much useful information for predicting high sales. I think this makes sense because discounted_price is closely related to other variables like original_price and discount_percentage, so the model may already be capturing that information through those variables.")
print("When comparing this model to the baseline, the overall performance is very similar. This shows that removing discounted_price did not hurt the model much, which means the simpler model is still effective. Overall, L1 regularization helped simplify the model while keeping similar predictive performance.")

Looking at the L1 model results, the model achieved an accuracy of about 0.75, which is very similar to the baseline model. The recall is relatively high at around 0.79, meaning the model does a good job identifying products that actually have high sales. However, the precision is lower at about 0.53, which suggests that when the model predicts high sales, it is not always correct.
One important thing about L1 regularization is that it removes less useful features by shrinking some coefficients to 0. In my model, the feature that was removed was discounted_price.
This suggests that discounted_price was not adding much useful information for predicting high sales. I think this makes sense because discounted_price is closely related to other variables like original_price and discount_percentage, so the model may already be capturing that information through those variables.
When comparing this model to the baseline, the overall performance is very similar. This shows that removing discou

In [65]:
l2_model = LogisticRegression(penalty="l2", max_iter=5000)
l2_model.fit(X_train_balanced_scaled, y_train_balanced)

y_pred_l2 = l2_model.predict(X_test_scaled)

print("L2 Accuracy:", accuracy_score(y_test, y_pred_l2))
print("L2 Precision:", precision_score(y_test, y_pred_l2))
print("L2 Recall:", recall_score(y_test, y_pred_l2))
print("L2 F1:", f1_score(y_test, y_pred_l2))

L2 Accuracy: 0.7560632688927944
L2 Precision: 0.5389308541545613
L2 Recall: 0.7893617021276595
L2 F1: 0.6405386740331491


In [66]:
# L2 Regularization Interpretation

print("Looking at the L2 model results, the model achieved an accuracy of about 0.756, which is slightly higher than the L1 model. The precision is around 0.539, recall is about 0.789, and the F1 score is approximately 0.641.")
print("L2 regularization works by shrinking coefficients but does not remove features completely like L1. This means all variables are still included in the model, but less important ones have smaller impact.")
print("Compared to the L1 model, the L2 model performed slightly better overall, especially in accuracy and F1 score. This suggests that keeping all features helped improve the model’s performance slightly.")
print("Overall, L2 provides a good balance by reducing overfitting while still using all available information from the dataset.")

Looking at the L2 model results, the model achieved an accuracy of about 0.756, which is slightly higher than the L1 model. The precision is around 0.539, recall is about 0.789, and the F1 score is approximately 0.641.
L2 regularization works by shrinking coefficients but does not remove features completely like L1. This means all variables are still included in the model, but less important ones have smaller impact.
Compared to the L1 model, the L2 model performed slightly better overall, especially in accuracy and F1 score. This suggests that keeping all features helped improve the model’s performance slightly.
Overall, L2 provides a good balance by reducing overfitting while still using all available information from the dataset.


In [67]:
from sklearn.linear_model import LogisticRegression

elastic_model = LogisticRegression(
    penalty="elasticnet",
    solver="saga",
    l1_ratio=0.5,
    max_iter=5000
)

elastic_model.fit(X_train_balanced_scaled, y_train_balanced)
y_pred_elastic = elastic_model.predict(X_test_scaled)

print("Elastic Net Accuracy:", accuracy_score(y_test, y_pred_elastic))
print("Elastic Net Precision:", precision_score(y_test, y_pred_elastic))
print("Elastic Net Recall:", recall_score(y_test, y_pred_elastic))
print("Elastic Net F1:", f1_score(y_test, y_pred_elastic))

Elastic Net Accuracy: 0.7528998242530756
Elastic Net Precision: 0.5347162201094785
Elastic Net Recall: 0.7897872340425532
Elastic Net F1: 0.6376911183645422


In [70]:
# Elastic Net Interpretation

print("Looking at the Elastic Net model results, the model achieved an accuracy of about 0.753, with a precision of around 0.535, recall of about 0.790, and an F1 score of approximately 0.638.")
print("Elastic Net combines both L1 and L2 regularization, meaning it can both shrink coefficients and remove less important features if needed. However, in this case, the performance is almost identical to the L1 model.")
print("This suggests that combining L1 and L2 did not significantly improve the model compared to using just one method. The dataset already seems well structured, and most features are contributing useful information.")
print("Overall, Elastic Net provides a balanced approach, but it does not outperform the simpler models in this case.")

Looking at the Elastic Net model results, the model achieved an accuracy of about 0.753, with a precision of around 0.535, recall of about 0.790, and an F1 score of approximately 0.638.
Elastic Net combines both L1 and L2 regularization, meaning it can both shrink coefficients and remove less important features if needed. However, in this case, the performance is almost identical to the L1 model.
This suggests that combining L1 and L2 did not significantly improve the model compared to using just one method. The dataset already seems well structured, and most features are contributing useful information.
Overall, Elastic Net provides a balanced approach, but it does not outperform the simpler models in this case.


In [71]:
# Final Model Comparison

print("Comparing all models, the baseline, L1, L2, and Elastic Net models all produced very similar results, with only small differences in accuracy and F1 score.")
print("The L1 model slightly simplified the model by removing the discounted_price feature, showing it was not very useful for predicting high sales. The L2 model kept all features and achieved the best overall performance, with the highest accuracy and F1 score, although the improvement was small.")
print("The Elastic Net model performed almost identically to the L1 model, showing that combining both regularization methods did not significantly change the results.")
print("Overall, this suggests that most features in the dataset are useful and the model is already well optimized. The L2 model is the best choice because it provides slightly better performance while keeping all features.")

Comparing all models, the baseline, L1, L2, and Elastic Net models all produced very similar results, with only small differences in accuracy and F1 score.
The L1 model slightly simplified the model by removing the discounted_price feature, showing it was not very useful for predicting high sales. The L2 model kept all features and achieved the best overall performance, with the highest accuracy and F1 score, although the improvement was small.
The Elastic Net model performed almost identically to the L1 model, showing that combining both regularization methods did not significantly change the results.
Overall, this suggests that most features in the dataset are useful and the model is already well optimized. The L2 model is the best choice because it provides slightly better performance while keeping all features.
